In [1]:
import xarray as xr
import os
from eofs.standard import Eof
import numpy as np

# from functions.SIT_functions.SIT_eddy_feedback_functions import eof_calc_alt

In [2]:
exp_type = 'cmip'
output_eof_file = '/home/users/cturrell/documents/eddy_feedback/b-parameter/cmip6_b/hist_NaN_issues/data'
force_eof_recalculate = True
pfull_slice = slice(850., 100.)
subtract_annual_cycle=True
eof_vars = ['ucomp', 'div1_QG', 'div1_QG_123', 'div1_QG_gt3']
n_eofs = 3
season_month_dict = {'DJF':[12,1,2,]}
propogate_all_nans = True

In [3]:
cnrm_path = '/gws/ssde/j25a/arctic_connect/cturrell/CMIP6/historical/CNRM-CM6-1-HR/1950_2015/6hrPlevPt/'
anom_ds_cnrm = xr.open_dataset(os.path.join(cnrm_path, '1979_2015', 'anoms.nc'))
# dataset_cnrm = read_daily_averages(os.path.join(cnrm_path, 'yearly_data'), start_month=1979, end_month=2015, exp_type='cmip6')

# eof_cnrm = eof_calc('cmip', output_eof_file, force_eof_recalculate, dataset_cnrm, pfull_slice,
#                               subtract_annual_cycle, eof_vars, n_eofs, season_month_dict, anom_ds_cnrm,
#                               propogate_all_nans, level_subset=[250., 500., 850.],
#                               pressure_weighted=True)

print(anom_ds_cnrm.isnull().sum())
anom_ds_cnrm

<xarray.Dataset> Size: 64B
Dimensions:           ()
Data variables:
    ucomp_anom        int64 8B 210128
    ucomp_orig        int64 8B 210128
    div1_QG_anom      int64 8B 236775
    div1_QG_orig      int64 8B 236775
    div1_QG_123_anom  int64 8B 10825368
    div1_QG_123_orig  int64 8B 10825368
    div1_QG_gt3_anom  int64 8B 10825368
    div1_QG_gt3_orig  int64 8B 10825368


<xarray.Dataset> Size: 2GB
Dimensions:           (lat: 360, pfull: 6, time: 13141)
Coordinates:
  * lat               (lat) float64 3kB -89.62 -89.12 -88.62 ... 89.12 89.62
  * pfull             (pfull) float32 24B 925.0 850.0 700.0 600.0 500.0 250.0
  * time              (time) datetime64[ns] 105kB 1979-01-01 ... 2015-01-01
Data variables:
    ucomp_anom        (time, pfull, lat) float64 227MB nan nan ... 0.6869 0.3173
    ucomp_orig        (time, pfull, lat) float32 114MB nan nan ... 1.582 0.6957
    div1_QG_anom      (time, pfull, lat) float64 227MB nan nan ... -0.5103
    div1_QG_orig      (time, pfull, lat) float64 227MB nan nan ... -0.1255
    div1_QG_123_anom  (time, pfull, lat) float64 227MB nan nan ... -0.5077
    div1_QG_123_orig  (time, pfull, lat) float64 227MB nan nan ... -0.1246
    div1_QG_gt3_anom  (time, pfull, lat) float64 227MB nan nan ... -0.002563
    div1_QG_gt3_orig  (time, pfull, lat) float64 227MB nan nan ... -0.0008538

In [4]:
def propagate_missing_data_to_all_vars(anom_ds, eof_vars):
    '''The EOF projectfield method requires that each of the variables
    we project must have missing data in the same place as the field 
    we are projecting onto. This is quite difficult to ensure
    particularly when I have added nans to div1 and div2 when the dtheta/dp
    is vanishingly small. To get around this, this function looks where the 
    nan values are in each of the anomaly fields and makes sure every other 
    variable also has missing data there.'''
    
    import numpy as np
    import logging

    anom_coord_list = [key for key in anom_ds.coords.keys()]
    anom_var_list = [f'{key}_anom' for key in eof_vars]

    n_nans_list = []

    for anom_var in anom_var_list:
        for anom_var_nan_var in anom_var_list:
            anom_ds[anom_var] = anom_ds[anom_var].where(np.isfinite(anom_ds[anom_var_nan_var]))
        n_nans_list.append(np.where(np.isnan(anom_ds[anom_var]))[0].shape[0])

    all_anom_vars_same_number_nan = np.all(np.asarray(n_nans_list)==n_nans_list[0])

    if all_anom_vars_same_number_nan:
        logging.info(f'successfully propagated missing data from all vars to all other vars to enable eof projection, so each field now has {n_nans_list[0]} nans')
    else:
        raise ValueError('anom vars contain differing number of nans, which means the projection will not work')
    
    return anom_ds

anom_prop = propagate_missing_data_to_all_vars(anom_ds_cnrm, eof_vars=eof_vars)
anom_prop

<xarray.Dataset> Size: 2GB
Dimensions:           (lat: 360, pfull: 6, time: 13141)
Coordinates:
  * lat               (lat) float64 3kB -89.62 -89.12 -88.62 ... 89.12 89.62
  * pfull             (pfull) float32 24B 925.0 850.0 700.0 600.0 500.0 250.0
  * time              (time) datetime64[ns] 105kB 1979-01-01 ... 2015-01-01
Data variables:
    ucomp_anom        (time, pfull, lat) float64 227MB nan nan ... 0.6869 0.3173
    ucomp_orig        (time, pfull, lat) float32 114MB nan nan ... 1.582 0.6957
    div1_QG_anom      (time, pfull, lat) float64 227MB nan nan ... -0.5103
    div1_QG_orig      (time, pfull, lat) float64 227MB nan nan ... -0.1255
    div1_QG_123_anom  (time, pfull, lat) float64 227MB nan nan ... -0.5077
    div1_QG_123_orig  (time, pfull, lat) float64 227MB nan nan ... -0.1246
    div1_QG_gt3_anom  (time, pfull, lat) float64 227MB nan nan ... -0.002563
    div1_QG_gt3_orig  (time, pfull, lat) float64 227MB nan nan ... -0.0008538

In [ ]:
def vert_integrate(data_array, pressure_weighted=True):
    if pressure_weighted:
        
        # Old method:
        # vert_int = data_array.integrate(coord='pfull') / data_array['pfull'].integrate(coord='pfull')

        # Use weighted mean with pfull as weights, which skips NaN values.
        # xarray.integrate uses trapezoidal rule and propagates NaN to the
        # whole column if any level is NaN, causing EOF failures after NaN
        # propagation across variables.
        
        # THIS METHOD IS NOT IDEAL FOR UNEVENLY SPACED DIMS!!!!!!!
        
        vert_int = data_array.weighted(data_array['pfull']).mean('pfull')
    else:
        vert_int = data_array.mean('pfull', skipna=True)

    return vert_int

anom_vert = vert_integrate(anom_prop)
anom_vert

<xarray.Dataset> Size: 284MB
Dimensions:           (lat: 360, time: 13141)
Coordinates:
  * lat               (lat) float64 3kB -89.62 -89.12 -88.62 ... 89.12 89.62
  * time              (time) datetime64[ns] 105kB 1979-01-01 ... 2015-01-01
Data variables:
    ucomp_anom        (time, lat) float64 38MB nan nan nan ... 0.004497 0.002164
    ucomp_orig        (time, lat) float32 19MB nan nan nan ... 0.005586 0.002631
    div1_QG_anom      (time, lat) float64 38MB nan nan nan ... 0.0075 0.003653
    div1_QG_orig      (time, lat) float64 38MB nan nan nan ... 0.006308 0.003644
    div1_QG_123_anom  (time, lat) float64 38MB nan nan nan ... 0.007473 0.003638
    div1_QG_123_orig  (time, lat) float64 38MB nan nan nan ... 0.006286 0.003628
    div1_QG_gt3_anom  (time, lat) float64 38MB nan nan ... 2.706e-05 1.563e-05
    div1_QG_gt3_orig  (time, lat) float64 38MB nan nan ... 2.22e-05 1.588e-05

In [6]:
def eof_calc_alt(data,lats):

    coslat = np.cos(np.deg2rad(lats)).clip(0., 1.)
    wgts = np.sqrt(coslat)[np.newaxis, np.newaxis, :]

    solver = Eof(data, weights=wgts, center=True)

    eofs = solver.eofsAsCovariance(neofs=3)
    pc1 = solver.pcs(npcs=3, pcscaling=1)

    variance_fractions = solver.varianceFraction(neigs=3)

    return eofs, pc1, variance_fractions, solver

eofs_va, pc1_va, variance_fractions_va, solver_va = eof_calc_alt(anom_vert.ucomp_anom.values, anom_vert.lat.values)

In [7]:
eofs_va

array([[            nan,             nan,             nan, ...,
         4.61797433e-06, -3.06112881e-08, -8.65039422e-07],
       [            nan,             nan,             nan, ...,
        -8.13064799e-05, -5.24644924e-05, -2.31373590e-05],
       [            nan,             nan,             nan, ...,
         3.51695336e-03,  2.27627959e-03,  1.00008806e-03]])